# Lesson 16: Database — Storing Data

## Why Do We Need a Database?

In previous lessons, when an agent created an article, the result appeared on screen and **disappeared**. Close the program, everything is lost.

In practice, we need to:
- **Store** articles that have been created
- **Track** status (processing, completed, error...)
- **Search** and **filter** articles by criteria
- **Review** edit history

A **database** handles all of this.

## Airtable — A Cloud Spreadsheet Database

Our product uses **Airtable** as its primary database — a smart spreadsheet in the cloud.

Comparison:
- **Without database**: Data lost when program closes
- **With Airtable**: Data saved in the cloud, accessible from anywhere, with a spreadsheet UI to browse

Why Airtable?
- **No setup file** — no `.db` file to manage or lose
- **Spreadsheet UI** — view and edit articles in a browser, like Google Sheets
- **Cloud-accessible** — access from any device, share with your team
- **API access** — Python can read/write data programmatically via `pyairtable`
- **Real-time** — changes appear instantly in the web interface

In [ ]:
from dotenv import load_dotenv
load_dotenv()

# Airtable uses a REST API — no local files needed
# The pyairtable library wraps this API for Python

# To use Airtable, you need two environment variables:
#   AIRTABLE_PAT      — Personal Access Token (like an API key)
#   AIRTABLE_BASE_ID  — The ID of your Airtable base (like a database name)

import os
pat = os.getenv("AIRTABLE_PAT")
base_id = os.getenv("AIRTABLE_BASE_ID")

print(f"AIRTABLE_PAT: {'Set' if pat else 'Not set'}")
print(f"AIRTABLE_BASE_ID: {'Set' if base_id else 'Not set'}")
print()
if pat and base_id:
    print("Airtable is configured! You can run the examples below.")
else:
    print("Airtable is not configured. Set these in your .env file.")
    print("You can still read through the code to understand the patterns.")

## Airtable Concepts

Airtable organizes data similarly to a spreadsheet:

| Airtable Concept | Spreadsheet Equivalent | SQL Equivalent |
|---|---|---|
| **Base** | A workbook (file) | Database |
| **Table** | A sheet/tab | Table |
| **Field** | A column | Column |
| **Record** | A row | Row |
| **Record ID** | (no equivalent) | Primary key |

### Record IDs — Strings, Not Numbers

In SQLite, each row gets an auto-incrementing integer ID: `1, 2, 3, ...`

In Airtable, each record gets a **string ID** like `"recABC123xyz"`. These are:
- Always strings (not integers)
- Generated by Airtable automatically
- Globally unique
- Used to reference records in API calls

This is why article IDs in our product are **strings**, not numbers. When you see `article_id = "recABC123"`, that's an Airtable record ID.

### pyairtable — The Python Library

We use `pyairtable` to interact with Airtable from Python:

```python
from pyairtable import Api

api = Api(os.getenv("AIRTABLE_PAT"))
table = api.table(os.getenv("AIRTABLE_BASE_ID"), "Articles")

# Create a record
record = table.create({"Topic": "SEO Guide", "Status": "queued"})
print(record["id"])  # "recABC123xyz"

# Read a record
record = table.get(record_id)
print(record["fields"]["Topic"])  # "SEO Guide"

# Update a record
table.update(record_id, {"Status": "review", "Word Count": 2000})

# List all records
records = table.all()
```

The pattern is simple: `table.create()`, `table.get()`, `table.update()`, `table.all()`.

## Airtable vs SQLite

You may encounter SQLite in other projects, so here's a quick comparison:

| Feature | SQLite | Airtable |
|---|---|---|
| Storage | Local `.db` file | Cloud (Airtable servers) |
| Access | Python only (or SQL tools) | Browser UI + API |
| Setup | `import sqlite3` (built-in) | Need API key + base ID |
| IDs | Integers (`1, 2, 3`) | Strings (`"recABC123"`) |
| Queries | SQL (`SELECT * FROM ...`) | API methods (`.all()`, `.get()`) |
| Sharing | Copy the file | Share a link |
| Cost | Free | Free tier (1000 records) |

**SQLite** is still used in our project for one thing: **chat session memory** in `chat.py`. Agno's built-in chat storage uses SQLite (`chat_sessions.db`). But for article data, Airtable is the primary database.

In [ ]:
# Let's see the Airtable functions in action
# These are mock examples showing the pattern — they work if Airtable is configured

import sys, os
sys.path.insert(0, os.path.abspath("../../output"))

# Show what functions our Airtable module provides
from tools.airtable import (
    create_article,
    get_article,
    list_articles,
    update_article_status,
)

print("Available Airtable functions:")
print("  create_article(topic, keywords)  — Create a new article (returns string ID)")
print("  get_article(article_id)          — Get one article as a dict")
print("  list_articles()                  — Get all articles as a list of dicts")
print("  update_article_status(id, status, **fields)  — Update status + any fields")
print()
print("Article IDs are strings like 'recABC123' (Airtable record IDs)")
print("Not integers like 1, 2, 3 (SQLite auto-increment)")

## The Real Product Database

The `tools/airtable.py` file provides these functions:
- `create_article(topic, keywords)` — Create a new article (status = 'queued'), returns string ID
- `get_article(id)` — View details of one article (returns a dict)
- `list_articles()` — View all articles
- `update_article_status(id, status, **fields)` — Update status and any other fields

The `**fields` pattern lets you update any combination of fields in one call:
```python
# Update just the status
update_article_status(article_id, "writing")

# Update status + content fields
update_article_status(article_id, "review", word_count=2000, article_markdown="# Title\n...")
```

The Airtable **Articles** table has these fields:
- `Topic` — the article topic
- `Target Keywords` — comma-separated keywords
- `Status` — current pipeline status
- `Outline JSON` — the structured outline from the Outline Agent
- `Article Markdown` — the full article content
- `Output File` — path to the saved .md file
- `Word Count` — article length
- `Meta Description` — SEO meta description
- `Error Message` — error details if the pipeline failed
- `Published URL` — where the article lives online (for rank tracking)
- `SQLite ID` — used as a unique key for upserts

The pipeline calls these functions to track progress:
```
queued -> researching -> outlining -> writing -> enriching -> review -> published (manual)
```

In [ ]:
# Create a test article via the Airtable layer
article_id = create_article("Test Article from Notebook", target_keywords=["test", "notebook"])
print(f"Created article {article_id}")
print(f"  (Notice: ID is a string like 'recXXX', not an integer)")

# Read it back
article = get_article(article_id)
print(f"\nArticle details:")
print(f"  Topic: {article['topic']}")
print(f"  Status: {article['status']}")
print(f"  ID: {article['id']}")

# Update status
update_article_status(article_id, "review", word_count=1000, article_markdown="# Test\n\nThis is a test.")
article = get_article(article_id)
print(f"\nAfter update:")
print(f"  Status: {article['status']}")
print(f"  Words: {article['word_count']}")

In [ ]:
# View all articles in Airtable
articles = list_articles()
print(f"Total articles: {len(articles)}\n")
for a in articles:
    print(f"  {a['id']}: {a['topic']} ({a['status']})")

## Rankings Table — Tracking SERP Positions

After publishing an article, you want to know: **is it ranking on Google?**

The **Rankings** table in Airtable records keyword positions over time. Each record is one check: "On this date, for this keyword, we ranked at position X."

Rankings table fields:
- `Keyword` — the search term being tracked
- `Position` — the SERP position (1 = top result)
- `URL` — the URL that ranked
- `Check Date` — when the check was performed
- `Search Engine` — typically "google"
- `Location` — geographic location for the search
- `Articles` — linked to the Articles table

This is how SEO professionals track progress — checking the same keywords weekly to see if positions improve.

The `tools/rankings.py` file provides `check_article_rankings()` which:
1. Queries the DataForSEO SERP API for each keyword
2. Saves results to the Rankings table in Airtable
3. Requires `DATA_FOR_SEO_API_KEY` and a `published_url` set on the article

## Summary

- A **database** stores data persistently — it survives program restarts
- **Airtable** is our primary database — cloud-hosted, spreadsheet UI, API access
- **Record IDs are strings** (`"recABC123"`) not integers (`1, 2, 3`)
- **4 key functions**: `create_article()`, `get_article()`, `list_articles()`, `update_article_status()`
- **Status tracking**: the database tracks each article through the pipeline (`queued` -> `researching` -> `writing` -> `enriching` -> `review`)
- **Rankings table** tracks SERP positions over time for published articles
- **SQLite** is only used for Agno chat memory (`chat_sessions.db`), not for articles

**Next lesson:** Tracing the journey of a request from chat to pipeline to Airtable.

In [ ]:
# Browsing the Airtable UI
print("You can also view and edit articles directly in Airtable:")
print()
print(f"  1. Go to https://airtable.com")
print(f"  2. Open your base (AIRTABLE_BASE_ID: {os.getenv('AIRTABLE_BASE_ID', 'not set')})")
print(f"  3. You'll see the Articles table with all articles")
print(f"  4. Click any row to see its full details")
print()
print("The Airtable UI shows the same data as the Python API.")
print("Changes in Airtable are reflected in the API and vice versa.")
print()
print("This is the key advantage over SQLite — you get a free,")
print("visual interface for browsing and editing your data.")

## Exercise

Using the Airtable functions imported above, write code that:

1. Creates a new article with a topic and keywords of your choice
2. Reads the article back and prints its topic, status, and ID
3. Updates the article status to `"review"` with a word count of 1500
4. Reads the article again and verifies the update

This is the same pattern the pipeline uses — create, update, read.

In [ ]:
# Exercise: Fill in the blanks to work with the Airtable database

# 1. Create an article (fill in topic and keywords)
my_id = create_article(___, target_keywords=[___, ___])
print(f"Created article: {my_id}")

# 2. Read the article back
my_article = ___(my_id)
print(f"Topic: {my_article[___]}")
print(f"Status: {my_article[___]}")
print(f"ID: {my_article['id']}")

# 3. Update the status to "review" with word_count=1500
update_article_status(my_id, ___, word_count=___)

# 4. Read again and verify
my_article = get_article(my_id)
print(f"\nAfter update:")
print(f"Status: {my_article[___]}")
print(f"Word count: {my_article[___]}")

<details>
<summary>Click to reveal answer</summary>

```python
# 1. Create an article
my_id = create_article("SEO Tips for Small Businesses", target_keywords=["seo tips", "small business"])
print(f"Created article: {my_id}")

# 2. Read the article back
my_article = get_article(my_id)
print(f"Topic: {my_article['topic']}")
print(f"Status: {my_article['status']}")
print(f"ID: {my_article['id']}")

# 3. Update the status to "review" with word_count=1500
update_article_status(my_id, "review", word_count=1500)

# 4. Read again and verify
my_article = get_article(my_id)
print(f"\nAfter update:")
print(f"Status: {my_article['status']}")
print(f"Word count: {my_article['word_count']}")
```
</details>